In [1]:
import pandas as pd
import plotly.express as px
from jupyter_dash import JupyterDash
from dash import html, dcc, Input, Output, callback


In [2]:
df = pd.read_csv(r"C:\Users\HP\Downloads\New folder\dataset.csv")

# Keep relevant columns
df = df[['track_name', 'artists', 'popularity', 'danceability', 'energy', 'valence', 'tempo']].dropna().drop_duplicates()

# Clean column values
df['track_name'] = df['track_name'].str.title().str.strip()
df['artists'] = df['artists'].str.title().str.strip()

print("✅ Data Loaded Successfully!")
print(f"Total Songs: {len(df):,}")
print("Columns:", ", ".join(df.columns))

 #PART 2 — DATA ANALYSIS & VISUALIZATION
# -------------------------------------------------

# Basic statistics
print("\n🎵 Summary Statistics:")
print(df.describe()[['popularity', 'danceability', 'energy', 'valence', 'tempo']].round(2))

# --- PLOT 1: Popularity Distribution ---
fig1 = px.histogram(df, x='popularity', nbins=40, title='Popularity Distribution', color_discrete_sequence=['#1DB954'])
fig1.show()

# --- PLOT 2: Top 10 Artists ---
top_artists = df.groupby('artists')['popularity'].mean().nlargest(10).reset_index()
fig2 = px.bar(top_artists, y='artists', x='popularity', orientation='h',
              title='Top 10 Artists by Popularity', color='popularity', color_continuous_scale='Viridis')
fig2.show()

# --- PLOT 3: Energy vs Danceability ---
fig3 = px.scatter(df, x='danceability', y='energy', color='popularity', 
                  title='Danceability vs Energy', color_continuous_scale='Plasma', opacity=0.7)
fig3.show()
 


✅ Data Loaded Successfully!
Total Songs: 86,734
Columns: track_name, artists, popularity, danceability, energy, valence, tempo

🎵 Summary Statistics:
       popularity  danceability    energy   valence     tempo
count    86734.00      86734.00  86734.00  86734.00  86734.00
mean        34.63          0.56      0.64      0.47    122.20
std         19.85          0.18      0.26      0.26     30.09
min          0.00          0.00      0.00      0.00      0.00
25%         21.00          0.45      0.46      0.25     99.63
50%         35.00          0.57      0.68      0.45    122.04
75%         49.00          0.69      0.86      0.68    140.13
max        100.00          0.98      1.00      1.00    243.37


In [4]:
# ============================================
# 🎵 Spotify Trend Dashboard (CSV Version)
# ============================================

import pandas as pd
import numpy as np
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

# --------------------------------------------
# 1️⃣ Load Data and Preprocess
# --------------------------------------------
# Replace with your CSV file path
csv_path = "spotify_data.csv"

# Load CSV
df = pd.read_csv(r"C:\Users\HP\Downloads\New folder\dataset.csv")

# Keep only important columns
columns_needed = ['track_name', 'artists', 'popularity', 'danceability', 'energy', 'valence', 'tempo']
df = df[columns_needed].dropna().drop_duplicates(subset='track_name')

# Calculate "Trending Probability"
# Formula (example): combines popularity + energy + valence
df['trending_probability'] = (
    (df['popularity'] * 0.5) +
    (df['energy'] * 0.25 * 100) +
    (df['valence'] * 0.25 * 100)
) / 100
df['trending_probability'] = np.round(df['trending_probability'], 2)

# Summary statistics
summary = df.describe()[['popularity', 'danceability', 'energy', 'valence', 'tempo']]

print("✅ Data Loaded Successfully!")
print(f"Total Songs: {len(df)}")
print("Columns:", ', '.join(df.columns))
print("\n🎵 Summary Statistics:")
print(summary)

# --------------------------------------------
# 2️⃣ Dash App Layout
# --------------------------------------------
app = Dash(__name__)
app.title = "Spotify Song Trend Dashboard"

app.layout = html.Div([
    html.H1("🎧 Spotify Song Trend Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.Label("🔍 Select or Search a Song:"),
        dcc.Dropdown(
            id='song-dropdown',
            options=[{'label': i, 'value': i} for i in sorted(df['track_name'].unique())],
            placeholder="Select a song...",
            style={'width': '80%', 'margin': 'auto'}
        )
    ], style={'textAlign': 'center', 'marginBottom': '30px'}),

    html.Div(id='song-stats', style={'textAlign': 'center', 'fontSize': 18, 'marginBottom': '20px'}),

    dcc.Graph(id='feature-graph'),

    html.Div([
        html.H3("📊 Overall Feature Correlation Heatmap"),
        dcc.Graph(
            figure=px.imshow(
                df[['popularity', 'danceability', 'energy', 'valence', 'tempo']].corr(),
                text_auto=True,
                color_continuous_scale='viridis',
                title="Feature Correlation Heatmap"
            )
        )
    ])
])

# --------------------------------------------
# 3️⃣ Callbacks: Update Graph & Stats
# --------------------------------------------
@app.callback(
    [Output('feature-graph', 'figure'),
     Output('song-stats', 'children')],
    [Input('song-dropdown', 'value')]
)
def update_dashboard(selected_song):
    if not selected_song:
        fig = px.histogram(df, x='popularity', nbins=30,
                           title="Popularity Distribution of All Songs")
        return fig, "Select a song to see detailed analysis."

    song_data = df[df['track_name'] == selected_song].iloc[0]

    # Create radar-like bar chart for features
    features = ['popularity', 'danceability', 'energy', 'valence', 'tempo']
    fig = px.bar(
        x=features,
        y=[song_data[f] for f in features],
        labels={'x': 'Feature', 'y': 'Value'},
        title=f"Audio Feature Profile: {selected_song}"
    )

    # Compute song probability and stats
    prob = song_data['trending_probability'] * 100
    stats_text = (
        f"🎶 **{selected_song}** by *{song_data['artists']}*<br>"
        f"🔥 Trending Probability: **{prob:.2f}%**<br>"
        f"📈 Popularity: {song_data['popularity']} / 100<br>"
        f"💃 Danceability: {song_data['danceability'] * 100:.1f}%<br>"
        f"⚡ Energy: {song_data['energy'] * 100:.1f}%<br>"
        f"😊 Valence (Happiness): {song_data['valence'] * 100:.1f}%<br>"
        f"🎵 Tempo: {song_data['tempo']:.1f} BPM"
    )

    return fig, stats_text

# --------------------------------------------
# 4️⃣ Run App
# --------------------------------------------
if __name__ == '__main__':
    print("✅ Running Spotify Trend Dashboard...")
    print("👉 Open http://127.0.0.1:8050 in your browser to view it.")
    app.run(debug=False, port=8060)


✅ Data Loaded Successfully!
Total Songs: 73608
Columns: track_name, artists, popularity, danceability, energy, valence, tempo, trending_probability

🎵 Summary Statistics:
         popularity  danceability        energy      valence         tempo
count  73608.000000  73608.000000  73608.000000  73608.00000  73608.000000
mean      34.386493      0.558683      0.636944      0.46722    122.143961
std       19.193482      0.178512      0.258685      0.26407     30.170407
min        0.000000      0.000000      0.000000      0.00000      0.000000
25%       21.000000      0.445000      0.457000      0.24500     99.158500
50%       34.000000      0.573000      0.680000      0.45500    122.037500
75%       48.000000      0.690000      0.859000      0.68200    140.132250
max      100.000000      0.985000      1.000000      0.99500    243.372000
✅ Running Spotify Trend Dashboard...
👉 Open http://127.0.0.1:8050 in your browser to view it.
